In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/window_features.csv")

df.head()

,meter_id,window_id,mean,std,min,max,median,range,load_factor,peak_to_avg,cv,mean_abs_diff,std_diff,day_mean,night_mean,day_night_ratio,zero_pct,low_pct,max_low_streak,theft
0,BR02,0,0.006529,0.007243,0.00035,0.027600,0.002825,0.027250,0.236564,4.227186,1.109299,0.007322,0.010007,0.007329,0.005729,1.279273,0.000000,0.333333,5,0.0
1,BR02,1,0.000913,0.003028,0.00000,0.011170,0.000000,0.011170,0.081729,12.235509,3.317322,0.000486,0.002188,0.000000,0.001826,0.000000,0.916667,0.916667,22,0.0
2,BR02,2,0.004579,0.005430,0.00000,0.020250,0.001650,0.020250,0.226132,4.422202,1.185853,0.003646,0.005730,0.005567,0.003592,1.549884,0.125000,0.375000,6,0.0
3,BR02,3,0.000101,0.000333,0.00000,0.001209,0.000000,0.001209,0.083183,12.021666,3.316631,0.000053,0.000247,0.000000,0.000201,0.000000,0.916667,0.916667,22,0.0
4,BR02,4,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0,0.0


In [3]:
print(df.shape)
df['theft'].value_counts()

(7299, 20)


theft
0.0    6610
1.0     689
Name: count, dtype: int64

In [ ]:
# Define Features & Target

In [4]:
# Separate features and target
X = df.drop(columns=['meter_id', 'window_id', 'theft'])
y = df['theft']

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

X.head()

Feature shape: (7299, 17)
Target shape: (7299,)


,mean,std,min,max,median,range,load_factor,peak_to_avg,cv,mean_abs_diff,std_diff,day_mean,night_mean,day_night_ratio,zero_pct,low_pct,max_low_streak
0,0.006529,0.007243,0.00035,0.027600,0.002825,0.027250,0.236564,4.227186,1.109299,0.007322,0.010007,0.007329,0.005729,1.279273,0.000000,0.333333,5
1,0.000913,0.003028,0.00000,0.011170,0.000000,0.011170,0.081729,12.235509,3.317322,0.000486,0.002188,0.000000,0.001826,0.000000,0.916667,0.916667,22
2,0.004579,0.005430,0.00000,0.020250,0.001650,0.020250,0.226132,4.422202,1.185853,0.003646,0.005730,0.005567,0.003592,1.549884,0.125000,0.375000,6
3,0.000101,0.000333,0.00000,0.001209,0.000000,0.001209,0.083183,12.021666,3.316631,0.000053,0.000247,0.000000,0.000201,0.000000,0.916667,0.916667,22
4,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0


In [ ]:
#Train/Test Split

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain distribution:")
print(y_train.value_counts())

print("\nTest distribution:")
print(y_test.value_counts())

Train shape: (5839, 17)
Test shape: (1460, 17)

Train distribution:
theft
0.0    5288
1.0     551
Name: count, dtype: int64

Test distribution:
theft
0.0    1322
1.0     138
Name: count, dtype: int64


In [ ]:
# Fraud detection is imbalanced
# Without stratification → model becomes useless
# This ensures:
# fair evaluation
# stable metrics

In [ ]:
#Training Random Forest Baseline

In [6]:
from sklearn.ensemble import RandomForestClassifier

# Initialize model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

# Train
rf_model.fit(X_train, y_train)

print("Model trained ✅")

Model trained ✅


In [ ]:
#Model Evaluation

In [7]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Predictions
y_pred = rf_model.predict(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9602739726027397

Confusion Matrix:
[[1316    6]
 [  52   86]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.96      1.00      0.98      1322
         1.0       0.93      0.62      0.75       138

    accuracy                           0.96      1460
   macro avg       0.95      0.81      0.86      1460
weighted avg       0.96      0.96      0.96      1460



In [ ]:
# Performance Breakdown (what matters)
# Confusion Matrix
# [[1316    6]
#  [  52   86]]
# TN = 1316 → normal correctly identified
# FP = 6 → very low false alarms ✅
# FN = 52 → missed theft ⚠️
# TP = 86 → detected theft
# 🎯 Key Metrics
# Accuracy
# 0.96

# ✔ High—but not the focus

# Precision (theft)
# 0.93

# ✔ Excellent
# 👉 When model says “theft”, it’s usually correct

# ⚠️ Recall (theft) — MOST IMPORTANT
# 0.62

# 👉 This is the bottleneck

# Detects 62% of theft
# Misses 38% (52 cases)
# F1-score
# 0.75

# ✔ Decent balanc

# “The model achieves high precision (0.93), meaning detected theft cases are reliable, but recall is moderate (0.62), 
# indicating some theft cases are still missed. 
# This is expected in baseline models and can be improved through threshold tuning or ensemble methods.”

In [ ]:
#Improve Detection

In [26]:
# Get probabilities
y_prob = rf_model.predict_proba(X_test)[:, 1]

# Preview
y_prob[:10]

array([0.  , 0.04, 0.07, 0.45, 0.12, 0.01, 0.06, 0.  , 0.  , 0.16])

In [27]:
# Adjust threshold to improve recall  Improve via threshold tuning
import numpy as np

# Try lower threshold
threshold = 0.3
y_pred_custom = (y_prob >= threshold).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

[[1281   41]
 [  34  104]]
              precision    recall  f1-score   support

         0.0       0.97      0.97      0.97      1322
         1.0       0.72      0.75      0.73       138

    accuracy                           0.95      1460
   macro avg       0.85      0.86      0.85      1460
weighted avg       0.95      0.95      0.95      1460



In [25]:
# # Adjust threshold to improve recall  Improve via threshold tuning
# import numpy as np

# # Try lower threshold
# threshold = 0.4
# y_pred_custom = (y_prob >= threshold).astype(int)

# from sklearn.metrics import classification_report, confusion_matrix

# print(confusion_matrix(y_test, y_pred_custom))
# print(classification_report(y_test, y_pred_custom))

In [ ]:
#XGBoost Setup

In [ ]:
#pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [15]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

print("XGBoost trained ✅")

c:\Users\ADMIN_\Desktop\ElectricityTheftDetection\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [21:38:53] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost trained ✅


In [16]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Predictions
y_pred_xgb = xgb_model.predict(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

# Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

Accuracy: 0.9554794520547946

Confusion Matrix:
[[1308   14]
 [  51   87]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98      1322
         1.0       0.86      0.63      0.73       138

    accuracy                           0.96      1460
   macro avg       0.91      0.81      0.85      1460
weighted avg       0.95      0.96      0.95      1460



In [ ]:
#Apply Threshold to XGBoost

#Just like RF, XGB powerful with probability tuning.

In [23]:
# Probabilities
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Try threshold = 0.
threshold = 0.3
y_pred_xgb_custom = (y_prob_xgb >= threshold).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_xgb_custom))
print(classification_report(y_test, y_pred_xgb_custom))

[[1293   29]
 [  40   98]]
              precision    recall  f1-score   support

         0.0       0.97      0.98      0.97      1322
         1.0       0.77      0.71      0.74       138

    accuracy                           0.95      1460
   macro avg       0.87      0.84      0.86      1460
weighted avg       0.95      0.95      0.95      1460



In [ ]:
# FINAL DECISION 
# ✅ Random Forest
# threshold = 0.3
# ✅ XGBoost
# threshold = 0.3

In [ ]:
#Saving Models

In [29]:
import pickle

# Save Random Forest
with open("models/rf_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

# Save XGBoost
with open("models/xgb_model.pkl", "wb") as f:
    pickle.dump(xgb_model, f)

print("Models saved ✅")

Models saved ✅


In [ ]:
#save thresholds

In [30]:
thresholds = {
    "rf_threshold": 0.3,
    "xgb_threshold": 0.3
}

with open("models/thresholds.pkl", "wb") as f:
    pickle.dump(thresholds, f)

print("Thresholds saved ✅")

Thresholds saved ✅
